# Exploración inicial — Dataset Automotriz Tier 1

Análisis exploratorio del dataset de producción industrial. Objetivo: entender estructura, dimensiones y contenido antes de calcular KPIs.

In [1]:
import pandas as pd
from pathlib import Path


## 1. Definir ruta del dataset

In [2]:
ruta_datos = Path("../data/raw/dataset_automotriz")
print(ruta_datos.exists())

True


## 2. Verificar contenido de la carpeta

In [3]:
archivos_csv = list(ruta_datos.glob("*.csv"))
print(f"Encontré {len(archivos_csv)} archivos CSV")



Encontré 11 archivos CSV


In [4]:
for archivo in archivos_csv:
    print(archivo.name)

fact_paros.csv
dim_operadores.csv
fact_defectos.csv
dim_turnos.csv
fact_mantenimiento.csv
fact_produccion.csv
dim_materia_prima.csv
dim_maquinas.csv
dim_proveedores.csv
fact_inspeccion_calidad.csv
dim_productos.csv


## 3. Cargar el CSV de produccion

In [5]:
produccion = pd.read_csv(ruta_datos / "fact_produccion.csv", parse_dates=["fecha"])
produccion.shape

(12256, 16)

## 4. Inspección inicial del DataFrame 

In [6]:
produccion.head(5)

,id_produccion,fecha,id_turno,id_maquina,id_linea,id_operador,id_producto,id_materia_prima,tiempo_planeado_min,tiempo_operativo_min,tiempo_paros_min,velocidad_ppm,piezas_planeadas,piezas_producidas,piezas_buenas,piezas_defectuosas
0,PRD0000001,2024-05-01,T01,M01,L01,OP010,SKU-AUTO-007,MP010,480,471.9,8.1,78.9,599,620,607,13
1,PRD0000002,2024-05-01,T01,M02,L01,OP009,SKU-AUTO-007,MP001,480,474.5,5.5,85.8,652,678,675,3
2,PRD0000003,2024-05-01,T01,M03,L02,OP005,SKU-AUTO-005,MP001,480,453.7,26.3,107.3,815,811,789,22
3,PRD0000004,2024-05-01,T01,M04,L02,OP001,SKU-AUTO-007,MP007,480,470.7,9.3,102.7,780,805,793,12
4,PRD0000005,2024-05-01,T01,M05,L03,OP005,SKU-AUTO-008,MP003,480,444.2,35.8,71.2,540,526,518,8


## 5. Calidad de datos: tipos y nulos

In [7]:
produccion.info()

<class 'pandas.DataFrame'>
RangeIndex: 12256 entries, 0 to 12255
Data columns (total 16 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   id_produccion         12256 non-null  str           
 1   fecha                 12256 non-null  datetime64[us]
 2   id_turno              12256 non-null  str           
 3   id_maquina            12256 non-null  str           
 4   id_linea              12256 non-null  str           
 5   id_operador           12256 non-null  str           
 6   id_producto           12256 non-null  str           
 7   id_materia_prima      12256 non-null  str           
 8   tiempo_planeado_min   12256 non-null  int64         
 9   tiempo_operativo_min  12256 non-null  float64       
 10  tiempo_paros_min      12256 non-null  float64       
 11  velocidad_ppm         12256 non-null  float64       
 12  piezas_planeadas      12256 non-null  int64         
 13  piezas_producidas     12256

## 6. Estadísticas descriptivas

In [8]:
produccion.describe()

,fecha,tiempo_planeado_min,tiempo_operativo_min,tiempo_paros_min,velocidad_ppm,piezas_planeadas,piezas_producidas,piezas_buenas,piezas_defectuosas
count,12256,12256.0,12256.000000,12256.000000,12256.000000,12256.000000,12256.000000,12256.000000,12256.000000
mean,2025-05-05 16:27:24.908616,480.0,454.555516,25.444484,84.260721,639.880956,637.608029,623.279292,14.328737
min,2024-05-01 00:00:00,480.0,280.000000,0.000000,50.000000,380.000000,335.000000,329.000000,0.000000
25%,2024-11-01 18:00:00,480.0,444.700000,7.300000,72.900000,554.000000,550.000000,539.000000,8.000000
50%,2025-05-06 00:00:00,480.0,462.400000,17.600000,81.600000,620.000000,619.000000,606.000000,13.000000
75%,2025-11-07 00:00:00,480.0,472.700000,35.300000,95.600000,726.000000,719.000000,704.000000,18.000000
max,2026-05-10 00:00:00,480.0,480.000000,200.000000,121.000000,919.000000,967.000000,947.000000,86.000000
std,NaN,0.0,25.779961,25.779961,14.557664,110.638115,115.138031,110.939017,8.606064


## 7. Producción mensual de la planta

In [9]:
produccion['año_mes'] = produccion['fecha'].dt.to_period('M')
produccion_mensual = produccion.groupby('año_mes')['piezas_producidas'].sum().sort_index(ascending=True)
produccion_mensual.head(12)

año_mes
2024-05    327605
2024-06    313712
2024-07    336164
2024-08    324305
2024-09    323343
2024-10    321010
2024-11    321271
2024-12    328225
2025-01    328807
2025-02    290934
2025-03    324400
2025-04    319875
Freq: M, Name: piezas_producidas, dtype: int64

In [10]:
produccion_mensual.tail(12)

año_mes
2025-06    314620
2025-07    328094
2025-08    325616
2025-09    315999
2025-10    335893
2025-11    322848
2025-12    333230
2026-01    326285
2026-02    289622
2026-03    322137
2026-04    315204
2026-05    101447
Freq: M, Name: piezas_producidas, dtype: int64

In [11]:
filtrado = produccion[produccion["año_mes"] == pd.Period("2025-02", freq="M")]
filtrado["fecha"].nunique()

28

In [12]:
analisis_maquinas = produccion.groupby("id_maquina").agg({
    "piezas_producidas" : "sum",
    "piezas_defectuosas" : "sum"
})
analisis_maquinas['porcentaje_scrap'] = analisis_maquinas['piezas_defectuosas'] / analisis_maquinas['piezas_producidas'] * 100
analisis_maquinas['porcentaje_scrap'] = analisis_maquinas['porcentaje_scrap'].round(2)
print(analisis_maquinas.sort_values("piezas_producidas", ascending=False))

            piezas_producidas  piezas_defectuosas  porcentaje_scrap
id_maquina                                                         
M04                   1556738               39395              2.53
M03                   1544471               38777              2.51
M02                   1278264               26464              2.07
M01                   1217203               24822              2.04
M06                   1146529               23588              2.06
M05                   1071319               22567              2.11


In [13]:
produccion.columns

Index(['id_produccion', 'fecha', 'id_turno', 'id_maquina', 'id_linea',
       'id_operador', 'id_producto', 'id_materia_prima', 'tiempo_planeado_min',
       'tiempo_operativo_min', 'tiempo_paros_min', 'velocidad_ppm',
       'piezas_planeadas', 'piezas_producidas', 'piezas_buenas',
       'piezas_defectuosas', 'año_mes'],
      dtype='str')